In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver Challenge — DL & GenAI Project

**Roll No:** 23f2004827

## Objective
Predict the top-3 most likely correct answers (ranked) for multiple-choice
questions, evaluated using Mean Average Precision at 3 (MAP@3).

## Approach
Four models are built and compared:
1. TF-IDF + cosine similarity (baseline, no training)
2. Pretrained sentence-transformer embeddings + cosine similarity
3. Fine-tuned BERT (pretrained, fine-tuned for multiple-choice)
4. Feedforward neural network trained from scratch on TF-IDF features

Final predictions use a weighted ensemble of models 2–4, with weights
selected via grid search on a held-out validation set.

## 1. Data Loading
Loading train and test sets from the competition data.

In [7]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head(2)

Train shape: (2000, 8)
Test shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


## 2. Evaluation Metric
Implementing MAP@3 locally so we can evaluate models on our own
validation split before submitting to Kaggle. A quick sanity check
confirms the function behaves as expected on toy inputs.

In [8]:
def map_at_3(predictions, true_answers):
    
    scores = []
    for pred, true in zip(predictions, true_answers):
        pred_list = pred.split()
        if true in pred_list:
            rank = pred_list.index(true) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
    return np.mean(scores)


test_preds = ["A B C", "B A C", "C D A"]
test_true  = ["A", "A", "A"]
print("Sanity check (expect ~0.5):", map_at_3(test_preds, test_true))

Sanity check (expect ~0.5): 0.611111111111111


## 3. Model 1 — TF-IDF + Cosine Similarity (Baseline)
This baseline represents questions and answer options using TF-IDF vectors and ranks the options using cosine similarity.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_predict(df):
    
    option_cols = ['A', 'B', 'C', 'D', 'E']
    predictions = []

    
    all_text = df['prompt'].tolist()
    for col in option_cols:
        all_text += df[col].tolist()

    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    vectorizer.fit(all_text)

    prompt_vecs = vectorizer.transform(df['prompt'])

    for i in range(len(df)):
        option_texts = [df.iloc[i][col] for col in option_cols]
        option_vecs = vectorizer.transform(option_texts)
        sims = cosine_similarity(prompt_vecs[i], option_vecs)[0]  

        ranked_idx = np.argsort(sims)[::-1]  
        top3_letters = [option_cols[idx] for idx in ranked_idx[:3]]
        predictions.append(" ".join(top3_letters))

    return predictions


train_preds_model1 = tfidf_predict(train)
score_model1 = map_at_3(train_preds_model1, train['answer'].tolist())
print("Model 1 (TF-IDF) MAP@3 on train:", score_model1)

Model 1 (TF-IDF) MAP@3 on train: 0.27616666666666667
